# 🎯 Midway Arbiter LoRA — Resume Run

**Coder + reasoner are already trained** (1-epoch adapters saved to Drive). This notebook finishes the remaining work: retrain the **arbiter** on the expanded 9,707-sample dataset.

| Adapter | Base model | Dataset | Epochs | Status |
|---|---|---|---|---|
| coder | `Qwen2.5-Coder-7B-Instruct` | `combined_lora_dataset.jsonl` (9125) | 1 | ✅ done (on Drive) |
| reasoner | `Qwen2.5-Coder-7B-Instruct` | `reasoner_lora_dataset.jsonl` (3500) | 1 | ✅ done (on Drive) |
| **arbiter** | `DeepSeek-R1-Distill-Qwen-7B` | `arbiter_lora_dataset.jsonl` (9707) | **1** | ⏳ train now |

> **Why 1 epoch:** with 9,707 samples the arbiter converges fast (loss ≈ 0.4 by 9% of epoch 1); 2 epochs was overkill and risks overfit.
> **Use a GPU runtime.** Outputs land in `MyDrive/midway-lora/output/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE = '/content/drive/MyDrive/midway-lora'
OUT = os.path.join(DRIVE, 'output')
os.makedirs(OUT, exist_ok=True)
print('Output dir:', OUT)

In [ ]:
# Unsloth (QLoRA trainer) + the HF stack.
# If the plain `unsloth` wheel is missing, uncomment the git install below.
!pip install -q unsloth
!pip install -q transformers datasets accelerate peft trl bitsandbytes
# !pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
print('dependencies installed')

In [ ]:
# Clone the repo to get the trainer, config, and pre-generated datasets.
import os
if not os.path.isdir('/content/midway-pipeline'):
    !git clone --depth 1 https://github.com/FoggyGoofball/midway-pipeline.git
os.chdir('/content/midway-pipeline/lora generator')
print('cwd:', os.getcwd())

In [ ]:
# Sanity-check the datasets landed with the clone.
import os
for f in ['combined_lora_dataset.jsonl', 'reasoner_lora_dataset.jsonl', 'arbiter_lora_dataset.jsonl']:
    n = sum(1 for _ in open(f, encoding='utf-8')) if os.path.exists(f) else 0
    print(f'{f}: {n} samples')

## Train the arbiter (supreme arbiter)

Base: `DeepSeek-R1-Distill-Qwen-7B` (a true reasoning model) · dataset: 9,707 tribunal debate samples · 1 epoch.

> ⚠️ If R1's chat template misbehaves during training, swap `--base-model` for `unsloth/Qwen2.5-Coder-7B-Instruct` and rerun just this cell.

In [ ]:
!python lora_fine_tune.py --dataset arbiter_lora_dataset.jsonl --output lora_output_arbiter --base-model unsloth/DeepSeek-R1-Distill-Qwen-7B --epochs 1 --train-on-inputs false --save-steps 100

## 4. Save everything to Drive

In [ ]:
import shutil, os, zipfile

OUT = '/content/drive/MyDrive/midway-lora/output'
os.makedirs(OUT, exist_ok=True)

# Copy the arbiter adapter (small) AND its Ollama-ready GGUF dir.
src = os.path.abspath('lora_output_arbiter')
if os.path.isdir(src):
    shutil.copytree(src, os.path.join(OUT, 'lora_output_arbiter'), dirs_exist_ok=True,
                    ignore=shutil.ignore_patterns(
                        'model-*.safetensors', 'model.safetensors.index.json',
                        'checkpoint-*', '.cache', 'optimizer.pt', 'scheduler.pt',
                        'rng_state.pth', 'training_args.bin'))
    print('saved arbiter adapter')

gguf = os.path.abspath('lora_output_arbiter_gguf')
if os.path.isdir(gguf):
    shutil.copytree(gguf, os.path.join(OUT, 'lora_output_arbiter_gguf'), dirs_exist_ok=True)
    print('saved arbiter GGUF')
else:
    print('WARN: missing lora_output_arbiter_gguf (trainer GGUF export did not run)')

# Slim zip: adapter + GGUF only.
zip_path = os.path.join(OUT, 'midway_arbiter.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for name in ['lora_output_arbiter', 'lora_output_arbiter_gguf']:
        s = os.path.abspath(name)
        if not os.path.isdir(s):
            continue
        for root, _, files in os.walk(s):
            if 'checkpoint-' in root or '.cache' in root:
                continue
            for f in files:
                if f.startswith('model-') or f == 'model.safetensors.index.json':
                    continue
                full = os.path.join(root, f)
                z.write(full, os.path.relpath(full, os.path.dirname(s)))
print('zipped ->', zip_path)

## Deploy to the Steam Deck (Ollama)

The arbiter output dir contains `adapter_model.safetensors` + a merged GGUF. Copy the GGUF to the Deck and:

```bash
ollama create midway-arbiter-lora -f Modelfile.arbiter
```

Then point the pipeline at it via `MIDWAY_ARBITER_MODEL`.